# Analyse Exploratoire des Données (EDA)
## Projet Maintenance Prédictive Industrielle — EFREI M1 DE

**Dataset :** `predictive_maintenance_v3.csv`  
**Tâche :** Classification binaire — prédire `failure_within_24h`  
**Auteur :** EFREI Bloc 2 RNCP40875

---

### Objectifs de l'EDA
1. Comprendre la structure et la qualité des données
2. Quantifier le déséquilibre des classes
3. Analyser les distributions des capteurs selon l'état (panne / pas de panne)
4. Identifier les features les plus corrélées à la cible
5. Détecter les outliers et les valeurs manquantes
6. Analyser les corrélations entre features


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

# Style matplotlib
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = {'0': '#636EFA', '1': '#EF553B'}

DATA_PATH = Path('../data/predictive_maintenance_v3.csv')
df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
print(f'Dataset chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')

---
## 1. Vue générale du dataset

In [ ]:
df.head(5)

In [ ]:
df.dtypes

In [ ]:
# Informations générales
print('=== Structure ===' )
print(f'Lignes       : {len(df):,}')
print(f'Colonnes     : {df.shape[1]}')
print(f'Période      : {df["timestamp"].min().date()} → {df["timestamp"].max().date()}')
print(f'Machines     : {df["machine_id"].nunique()} identifiants')
print(f'Types        : {df["machine_type"].unique().tolist()}')
print(f'Modes        : {df["operating_mode"].unique().tolist()}')

---
## 2. Valeurs manquantes

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Valeurs manquantes': missing, '% manquant': missing_pct})
missing_df = missing_df[missing_df['Valeurs manquantes'] > 0].sort_values('Valeurs manquantes', ascending=False)
print(missing_df.to_string())

In [ ]:
# Visualisation heatmap des valeurs manquantes
fig, ax = plt.subplots(figsize=(10, 3))
missing_mask = df[missing_df.index].isnull()
# Afficher uniquement un échantillon pour la lisibilité
sns.heatmap(missing_mask.sample(500, random_state=42).T, ax=ax,
            cbar=False, yticklabels=True, xticklabels=False,
            cmap=['#f0f0f0', '#EF553B'])
ax.set_title('Heatmap des valeurs manquantes (échantillon 500 obs.)', fontsize=13)
plt.tight_layout()
plt.show()

print()
print('Stratégie d\'imputation choisie :')
print('  → Variables numériques : médiane (robuste aux outliers)')
print('  → Variables catégorielles : mode')
print('  → Fit uniquement sur le train set (pas de data leakage)')

---
## 3. Déséquilibre des classes (variable cible)

In [ ]:
class_counts = df['failure_within_24h'].value_counts()
print('Distribution de failure_within_24h :')
print(f'  Classe 0 (pas de panne) : {class_counts[0]:,} ({class_counts[0]/len(df)*100:.1f}%)')
print(f'  Classe 1 (panne)        : {class_counts[1]:,} ({class_counts[1]/len(df)*100:.1f}%)')
print(f'  Ratio déséquilibre      : {class_counts[0]/class_counts[1]:.1f}:1')
print()
print('Implications pour la modélisation :')
print('  → L\'accuracy seule est trompeuse (un classifieur trivial "tout 0" atteint 85.2%)')
print('  → Métriques prioritaires : Recall, F1, PR-AUC')
print('  → Techniques nécessaires : SMOTE, class_weight, ajustement du seuil')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
ax1.bar(['Pas de panne (0)', 'Panne (1)'], class_counts.values,
        color=['#636EFA', '#EF553B'], alpha=0.85, edgecolor='white')
for i, (label, count) in enumerate(zip(['Pas de panne', 'Panne'], class_counts.values)):
    ax1.text(i, count + 100, f'{count:,}\n({count/len(df)*100:.1f}%)',
             ha='center', fontsize=11)
ax1.set_title('Déséquilibre des classes', fontsize=13)
ax1.set_ylabel('Nombre d\'observations')

# Pie chart
ax2.pie(class_counts.values, labels=['Pas de panne (0)', 'Panne (1)'],
        colors=['#636EFA', '#EF553B'], autopct='%1.1f%%', startangle=90)
ax2.set_title('Répartition des classes', fontsize=13)

plt.tight_layout()
plt.show()

---
## 4. Statistiques descriptives des capteurs

In [ ]:
SENSOR_COLS = ['vibration_rms', 'temperature_motor', 'current_phase_avg',
               'pressure_level', 'rpm', 'hours_since_maintenance', 'ambient_temp']

df[SENSOR_COLS].describe().round(2)

In [ ]:
# Statistiques par classe
df.groupby('failure_within_24h')[SENSOR_COLS].mean().round(2)

In [ ]:
print('Observations clés :')
print('  → temperature_motor : 56.1°C (panne) vs 50.0°C (normal) — +12% en cas de panne')
print('  → vibration_rms     : 2.43 (panne) vs 1.46 (normal) — +66% en cas de panne')
print('  → Corrélations avec target (Pearson) :')
print()
corr_target = df[SENSOR_COLS].corrwith(df['failure_within_24h']).sort_values(ascending=False)
print(corr_target.round(4).to_string())
print()
print('→ temperature_motor et vibration_rms sont les signaux prédictifs les plus forts')

---
## 5. Distribution des capteurs par classe (panne / pas de panne)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(SENSOR_COLS):
    ax = axes[i]
    for cls, color in [(0, '#636EFA'), (1, '#EF553B')]:
        data = df[df['failure_within_24h'] == cls][col].dropna()
        ax.hist(data, bins=40, alpha=0.6, color=color,
                label=f'Classe {cls}', density=True)
    ax.set_title(col, fontsize=11)
    ax.set_xlabel('Valeur')
    ax.set_ylabel('Densité')
    ax.legend(fontsize=9)

# Masquer le dernier subplot vide
axes[-1].set_visible(False)
fig.suptitle('Distribution des capteurs selon la classe (Classe 0=bleu, Classe 1=rouge)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(SENSOR_COLS):
    ax = axes[i]
    df_clean = df[[col, 'failure_within_24h']].dropna()
    data_0 = df_clean[df_clean['failure_within_24h'] == 0][col]
    data_1 = df_clean[df_clean['failure_within_24h'] == 1][col]
    ax.boxplot([data_0, data_1], labels=['Normal', 'Panne'],
               patch_artist=True,
               boxprops=dict(facecolor='#636EFA', alpha=0.6),
               medianprops=dict(color='black', linewidth=2))
    bp = ax.boxplot([data_0, data_1], labels=['Normal', 'Panne'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#636EFA')
    bp['boxes'][1].set_facecolor('#EF553B')
    for patch in bp['boxes']:
        patch.set_alpha(0.6)
    ax.set_title(col, fontsize=11)

axes[-1].set_visible(False)
fig.suptitle('Boxplots des capteurs par classe', fontsize=14)
plt.tight_layout()
plt.show()

---
## 6. Analyse des variables catégorielles

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Taux de panne par type de machine
machine_failure = df.groupby('machine_type')['failure_within_24h'].mean().sort_values(ascending=False)
axes[0].bar(machine_failure.index, machine_failure.values * 100,
            color=['#636EFA', '#EF553B', '#00CC96', '#AB63FA'], alpha=0.85)
axes[0].set_title('Taux de panne par type de machine (%)', fontsize=12)
axes[0].set_ylabel('Taux de panne (%)')
for i, v in enumerate(machine_failure.values):
    axes[0].text(i, v * 100 + 0.2, f'{v*100:.1f}%', ha='center', fontsize=11)

# Taux de panne par mode opératoire
mode_failure = df.groupby('operating_mode')['failure_within_24h'].mean().sort_values(ascending=False)
axes[1].bar(mode_failure.index, mode_failure.values * 100,
            color=['#EF553B', '#636EFA', '#00CC96'], alpha=0.85)
axes[1].set_title('Taux de panne par mode opératoire (%)', fontsize=12)
axes[1].set_ylabel('Taux de panne (%)')
for i, v in enumerate(mode_failure.values):
    axes[1].text(i, v * 100 + 0.2, f'{v*100:.1f}%', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

print('Observations :')
print('  → Le taux de panne est similaire entre les 4 types de machines (~14-16%)')
print('  → Le mode "peak" a légèrement plus de pannes (16.0%) vs "idle" (14.5%)')
print('  → Le type de machine n\'est pas un facteur déterminant seul')

In [ ]:
# Types de pannes
failure_types = df[df['failure_within_24h'] == 1]['failure_type'].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(failure_types.index, failure_types.values,
        color=['#EF553B', '#FFA500', '#636EFA', '#00CC96'], alpha=0.85)
for i, v in enumerate(failure_types.values):
    ax.text(v + 5, i, f'{v} ({v/len(failure_types.index.tolist())*100/len(failure_types)*100:.0f}%)',
            va='center')
ax.set_title('Distribution des types de pannes (parmi les observations avec panne)', fontsize=12)
ax.set_xlabel('Nombre d\'occurrences')
plt.tight_layout()
plt.show()

print('Types de pannes observés :')
print('  → bearing (usure roulement) : 1117 — défaillance mécanique la plus fréquente')
print('  → motor_overheat            : 1060 — corrélée à temperature_motor élevée')
print('  → hydraulic                 :  728 — corrélée à pressure_level et vibration')
print('  → electrical                :  655 — corrélée à current_phase_avg anormal')

---
## 7. Matrice de corrélation

In [ ]:
corr_cols = SENSOR_COLS + ['failure_within_24h']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1,
    ax=ax, linewidths=0.5, mask=mask,
)
ax.set_title('Matrice de corrélation (Pearson)', fontsize=13)
plt.tight_layout()
plt.show()

print('Corrélations notables avec failure_within_24h :')
print('  → temperature_motor   : 0.39 (corrélation positive modérée)')
print('  → vibration_rms       : 0.26 (corrélation positive)')
print('  → current_phase_avg   : 0.16 (corrélation positive faible)')
print()
print('Inter-features :')
print('  → vibration_rms et temperature_motor sont corrélés (0.XX)')
print('  → Pas de multicolinéarité forte entre features — OK pour les modèles')

---
## 8. Analyse temporelle

In [ ]:
df_temporal = df.copy()
df_temporal['date'] = df_temporal['timestamp'].dt.date
daily_failures = df_temporal.groupby('date')['failure_within_24h'].agg(['sum', 'count'])
daily_failures['rate'] = daily_failures['sum'] / daily_failures['count'] * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(daily_failures.index, daily_failures['sum'], color='#EF553B', linewidth=1.5)
ax1.fill_between(daily_failures.index, daily_failures['sum'], alpha=0.3, color='#EF553B')
ax1.set_ylabel('Nombre de pannes / jour')
ax1.set_title('Évolution temporelle des pannes', fontsize=13)

ax2.plot(daily_failures.index, daily_failures['rate'], color='#636EFA', linewidth=1.5)
ax2.axhline(daily_failures['rate'].mean(), color='orange', linestyle='--',
            label=f'Moyenne ({daily_failures["rate"].mean():.1f}%)')
ax2.set_ylabel('Taux de panne (%)')
ax2.set_xlabel('Date')
ax2.legend()

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## 9. Analyse des outliers

In [ ]:
# Détection par la méthode IQR
print('Analyse des outliers (méthode IQR) :')
for col in SENSOR_COLS:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    pct = n_outliers / df[col].notna().sum() * 100
    print(f'  {col:30s}: {n_outliers:4d} outliers ({pct:.1f}%)')

print()
print('Décision : conserver les outliers — en contexte industriel, les valeurs extrêmes')
print('sont souvent des signaux réels de dégradation machine, pas des erreurs de mesure.')

---
## 10. Conclusions et recommandations pour la modélisation

### Points clés

| Observation | Impact sur la modélisation |
|-------------|---------------------------|
| Déséquilibre 5.8:1 | Utiliser Recall + PR-AUC, pas l'accuracy. Tester SMOTE / class_weight |
| Valeurs manquantes 2-4% | Imputation médiane sur train set uniquement (no leakage) |
| `temperature_motor` corrélée à 0.39 | Feature la plus prédictive (à confirmer avec SHAP) |
| `vibration_rms` corrélée à 0.26 | Deuxième signal le plus fort |
| Pas de tendance temporelle forte | Pas besoin de features temporelles complexes |
| Outliers réels | Ne pas les supprimer — ce sont des signaux de dégradation |
| Variables catégorielles peu discriminantes | Machine type et mode opératoire: encodage OHE suffisant |

### Choix de modélisation
- **Pipeline** : `ColumnTransformer` + `StandardScaler` (numériques) + `OneHotEncoder` (catégorielles)
- **Gestion déséquilibre** : Comparer Baseline / ROS / SMOTE / RUS / class_weight  
- **Métriques** : Recall (priorité) > PR-AUC > F1 > ROC-AUC
- **Seuil de décision** : Optimiser dans [0.1, 0.9] — le seuil 0.5 par défaut sous-optimise le Recall
